# 7. Spatial Agreement Analysis
Cross-tabulation and spatial agreement between AHP and Random Forest flood susceptibility classification maps.

In [ ]:
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

## Configuration / 

In [ ]:
# Paths to classified susceptibility rasters (classes 1-5)
AHP_PATH = r"D:\Jokian\Jurnal Pak Sandri\Jurnal_Tapanuli\3_Peta\Peta Perbandingan\ahp_classified.tif"
RF_PATH = r"D:\Jokian\Jurnal Pak Sandri\Jurnal_Tapanuli\3_Peta\Peta Perbandingan\rf_classified.tif"

# Pathlib Path for output directory
PROJECT_ROOT = Path.cwd().parent

LABELS = ['Very Low', 'Low', 'Moderate', 'High', 'Very High']

## 1. Load Rasters / 

In [ ]:
with rasterio.open(AHP_PATH) as src:
    ahp = src.read(1)
    profile = src.profile
    nodata_a = src.nodata

with rasterio.open(RF_PATH) as src:
    rf = src.read(1)
    nodata_r = src.nodata

# Mask valid pixels (classes 1-5)
valid = np.isin(ahp, [1,2,3,4,5]) & np.isin(rf, [1,2,3,4,5])
a = ahp[valid]
r = rf[valid]
n = valid.sum()
print(f'Valid pixels: {n:,}')

## 2. Class Distribution / 

In [ ]:
dist_data = {}
for k in range(1, 6):
    dist_data[LABELS[k-1]] = {
        'AHP (%)': round((a == k).sum() / n * 100, 1),
        'RF (%)': round((r == k).sum() / n * 100, 1)
    }

dist_df = pd.DataFrame(dist_data).T
print('Class Distribution (% Area):')
print(dist_df.to_string())

## 3. Agreement Matrix / 

Cross-tabulation of AHP (rows) vs RF (cols) classifications.

In [ ]:
matrix = np.zeros((5, 5))
for i in range(1, 6):
    for j in range(1, 6):
        matrix[i-1, j-1] = ((a == i) & (r == j)).sum() / n * 100

matrix_df = pd.DataFrame(matrix, index=[f'AHP-{l}' for l in LABELS],
    columns=[f'RF-{l}' for l in LABELS])
print('Agreement Matrix (% Area):')
print(matrix_df.round(2).to_string())

plt.figure(figsize=(8, 6))
sns.heatmap(matrix_df, annot=True, fmt='.1f', cmap='YlOrRd', linewidths=0.5)
plt.title('AHP vs RF Agreement Matrix (% Area)')
plt.xlabel('Random Forest')
plt.ylabel('AHP')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'outputs' / 'agreement_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Agreement Statistics / 

In [ ]:
exact_agree = np.trace(matrix)

within1 = sum(
 matrix[i, j]
 for i in range(5) for j in range(5)
 if abs(i - j) <= 1
)

rf_higher = sum(matrix[i, j] for i in range(5) for j in range(5) if j > i)
ahp_higher = sum(matrix[i, j] for i in range(5) for j in range(5) if i > j)

print('Agreement Statistics:')
print(f' Exact agreement: {exact_agree:.1f}%')
print(f' Agreement +/-1 class: {within1:.1f}%')
print(f' RF > AHP: {rf_higher:.1f}%')
print(f' AHP > RF: {ahp_higher:.1f}%')
print(f' High + Very High AHP: {(np.isin(a, [4,5])).sum() / n * 100:.1f}%')
print(f' High + Very High RF: {(np.isin(r, [4,5])).sum() / n * 100:.1f}%')
print(f' Moderate AHP: {(a == 3).sum() / n * 100:.1f}%')
print(f' Moderate RF: {(r == 3).sum() / n * 100:.1f}%')

## 5. Save Agreement Map / 

Creates a raster: 0=nodata, 1=agree, 2=AHP higher, 3=RF higher

In [ ]:
agree_map = np.full(ahp.shape, 0, dtype=np.uint8)
agree_map[valid & (ahp == rf)] = 1
agree_map[valid & (ahp > rf)] = 2
agree_map[valid & (ahp < rf)] = 3

out_profile = profile.copy()
out_profile.update(dtype=rasterio.uint8, nodata=0)
with rasterio.open(PROJECT_ROOT / 'outputs' / 'agreement_map.tif', 'w', **out_profile) as dst:
    dst.write(agree_map, 1)

print('Saved: outputs/agreement_map.tif')
print(' 1 = Agree, 2 = AHP Higher, 3 = RF Higher')

## 6. Save Statistics / 

In [ ]:
stats_df = pd.DataFrame({
    'Metric': ['Exact Agreement', 'Agreement +/-1 class', 'RF > AHP', 'AHP > RF'],
    'Value (%)': [exact_agree, within1, rf_higher, ahp_higher]
})
stats_df.to_csv(PROJECT_ROOT / 'outputs' / 'spatial_agreement.csv', index=False)
matrix_df.round(2).to_csv(PROJECT_ROOT / 'outputs' / 'agreement_matrix.csv')
print('Saved: outputs/spatial_agreement.csv, outputs/agreement_matrix.csv')